# Project 1: Data Analyst project


##### 1 - Inequality in Denmark

Opening the IFOR41 dataset from Statistics Denmark, which contains Gini coefficient data for the whole of Denmark over time. After importing the file, we clean the data by removing unnecessary rows and columns and assigning clear column names.

In [3]:
# 1 - Inequality in Denmark

%pip install git+https://github.com/alemartinello/dstapi
#%pip install fredapi


  Cloning https://github.com/alemartinello/dstapi to C:\Users\whoop\AppData\Local\Temp\pip-req-build-6lu1of63
  Resolved https://github.com/alemartinello/dstapi to commit d9eeb5a82cbc70b7d63b2ff44d92632fd77123a4
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Note: you may need to restart the kernel to use updated packages.


  Running command git clone --filter=blob:none --quiet https://github.com/alemartinello/dstapi 'C:\Users\whoop\AppData\Local\Temp\pip-req-build-6lu1of63'


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

# Plotting
import matplotlib.pyplot as plt
plt.rcParams.update({'axes.grid':True,'grid.color':'black','grid.alpha':'0.25','grid.linestyle':'--'})
plt.rcParams.update({'font.size': 14})

# Importerer API
from dstapi import DstApi
#from fredapi import Fred

### 1.1 The Gini coefficient and the top 10 percent share

In [10]:
# Load table IFOR41 from Statistics Denmark
IFOR41 = DstApi('IFOR41')

# Overview Table
IFOR41.tablesummary(language='en')




Table IFOR41: Income distribution on equivalised disposable income by indicator, municipality and time
Last update: 2025-12-01T08:00:00


,variable name,# values,First value,First value label,Last value,Last value label,Time variable
0,ULLIG,4,70,Gini coefficient,73,P90/10 (Upper decile boundary divided by lower),False
1,KOMMUNEDK,99,000,All Denmark,851,Aalborg,False
2,Tid,38,1987,1987,2024,2024,True


In [12]:
# More details - Gini coefficient
IFOR41.variable_levels('ULLIG', language='en')

,id,text
0,70,Gini coefficient
1,71,Hoover index
2,72,S80/S20 (Income quintile share ratio)
3,73,P90/10 (Upper decile boundary divided by lower)


In [13]:
IFOR41.variable_levels('KOMMUNEDK', language='en')

,id,text
0,000,All Denmark
1,101,Copenhagen
2,147,Frederiksberg
3,155,Dragør
4,185,Tårnby
...,...,...
94,773,Morsø
95,840,Rebild
96,787,Thisted
97,820,Vesthimmerlands


Nu prøver jeg at lave figuren


In [2]:
# Thongs kode

gini = IFOR41.get_data(
    params={
        'table': 'IFOR41',
        'format': 'BULK',
        'lang': 'en',
        'variables': [
            {
                'code': 'KOMMUNEDK',
                'values': ["000"]
            },
            {
                'code': 'ULLIG',
                'values': ["70"]
            },
            {
                'code': 'Tid',
                'values': ["*"]
            }
        ]
    }
)

print(gini.head())
print(gini.columns)

NameError: name 'IFOR41' is not defined

In [8]:
plt.figure(figsize=(18, 10))
plt.plot(GiniDenmark.iloc[:, 0], GiniDenmark.iloc[:, 1], marker='o')
plt.xticks(GiniDenmark.iloc[:, 0][::1], rotation=45)
plt.xlabel("Year")
plt.ylabel("Gini coefficient, Denmark %")
plt.title("Gini coefficient for all of Denmark over time")
plt.grid(True)
plt.show()

NameError: name 'GiniDenmark' is not defined

<Figure size 1800x1000 with 0 Axes>

In [7]:
# --------------------------------------------------
# Gini coefficient - IFOR41
# --------------------------------------------------

# Load table IFOR41 from Statistics Denmark
IFOR41 = DstApi('IFOR41')

# Overview Table
IFOR41.tablesummary(language='en')

# More details - Gini coefficient
IFOR41.variable_levels('ULLIG', language='en')


# Download Gini coefficient
params_gini = {
    'table': 'IFOR41',
    'format': 'BULK',
    'lang': 'en',
    'variables': [
        {'code': 'ULLIG', 'values': ['70']},
        {'code': 'KOMMUNE', 'values': ['000']},
        {'code': 'Tid', 'values': ['*']}
    ]
}

gini = IFOR41.get_data(params=params_gini)

# Clean Gini data
gini['TID'] = gini['TID'].astype(int)
gini['INDHOLD'] = gini['INDHOLD'].astype(float)

gini = gini[['TID', 'INDHOLD']]

gini = gini.rename(columns={
    'TID': 'year',
    'INDHOLD': 'gini'
})

gini = gini.sort_values('year').reset_index(drop=True)


# --------------------------------------------------
# Top 10 percent income share - IFOR32
# --------------------------------------------------

# Load table IFOR32
IFOR32 = DstApi('IFOR32')

# Overview Table
IFOR32.tablesummary(language='en')

# More details - deciles
IFOR32.variable_levels('DECIL', language='en')


# Download decile income data
params_deciles = {
    'table': 'IFOR32',
    'format': 'BULK',
    'lang': 'en',
    'variables': [
        {'code': 'DECIL', 'values': ['*']},
        {'code': 'KOMMUNE', 'values': ['000']},
        {'code': 'Tid', 'values': ['*']}
    ]
}

deciles = IFOR32.get_data(params=params_deciles)

# Clean decile data
deciles['TID'] = deciles['TID'].astype(int)
deciles['INDHOLD'] = deciles['INDHOLD'].astype(float)

deciles = deciles.rename(columns={
    'TID': 'year',
    'INDHOLD': 'income'
})


# Extract decile number
deciles['decile_number'] = (
    deciles['DECIL']
    .astype(str)
    .str.extract(r'(\d+)')
    .astype(int)
)


# --------------------------------------------------
# Calculate top 10 percent income share
# --------------------------------------------------

# Sum of average income across all 10 deciles
total_income = (
    deciles
    .groupby('year', as_index=False)['income']
    .sum()
    .rename(columns={'income': 'total_income'})
)

# Income in the 10th decile
top_decile = (
    deciles[deciles['decile_number'] == 10]
    [['year', 'income']]
    .rename(columns={'income': 'top_decile_income'})
)

# Merge
top10 = pd.merge(
    top_decile,
    total_income,
    on='year',
    validate='1:1'
)

# Top 10 percent income share
top10['top10_percent'] = (
    top10['top_decile_income']
    / top10['total_income']
    * 100
)


# --------------------------------------------------
# Merge Gini and top 10 percent share
# --------------------------------------------------

inequality = pd.merge(
    gini,
    top10[['year', 'top10_percent']],
    on='year',
    validate='1:1'
)

display(inequality.head())


# --------------------------------------------------
# Plot
# --------------------------------------------------

fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.plot(
    inequality['year'],
    inequality['gini']
)

ax1.set_xlabel('Year')
ax1.set_ylabel('Gini coefficient')


ax2 = ax1.twinx()

ax2.plot(
    inequality['year'],
    inequality['top10_percent'],
    linestyle='--'
)

ax2.set_ylabel('Top 10% income share (%)')

plt.title('Income inequality in Denmark')

fig.tight_layout()
plt.show()


# --------------------------------------------------
# Correlation
# --------------------------------------------------

correlation = inequality['gini'].corr(
    inequality['top10_percent']
)

print(f'Correlation: {correlation:.3f}')

Table IFOR41: Income distribution on equivalised disposable income by indicator, municipality and time
Last update: 2025-12-01T08:00:00


KeyError: 'TID'

In [ ]:
print(gini.columns)
display(gini.head())

Index(['{"errorTypeCode":"EXTRACT-NOTALLOWED","message":"Values must be selected for variable: KOMMUNEDK"}'], dtype='str')


,"{""errorTypeCode"":""EXTRACT-NOTALLOWED"",""message"":""Values must be selected for variable: KOMMUNEDK""}"


In [ ]:
# --------------------------------------------------
# 2. Top 10 percent income share - IFOR32
# --------------------------------------------------

# Load table IFOR32
IFOR32 = DstApi('IFOR32')

# Overview of table
IFOR32.tablesummary(language='en')

# Inspect decile variable
IFOR32.variable_levels('DECIL', language='en')

Table IFOR32: Avg. equivalised disposable Income in decile groups, by decile average, municipality and time
Last update: 2025-12-01T08:00:00

                Error: The table does not seem to contain the requested variable.
                Check the spelling (variable names are case sensitive
                )


IndexError('list index out of range')

### 1.2 Prediction

In [6]:
# Import minimize
from scipy.optimize import minimize

# t is year after 1987
gini_api["t"] = gini_api["year"] - 1987

t = gini_api["t"].to_numpy()
y = gini_api["gini"].to_numpy()

TypeError: 'DstApi' object is not subscriptable